In [1]:
import requests
from time import sleep, strftime
import random
from random import randint

from bs4 import BeautifulSoup 

In [2]:
!pip install selenium

In [3]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
selenium.__version__

'4.38.0'

In [4]:
options = webdriver.FirefoxOptions()
navegador = webdriver.Firefox(options=options)

In [5]:
#sleep(randint(2,4))
navegador.get("https://www.aeropuertobarcelona-elprat.com/cast/salidas-aeropuerto-barcelona.html?t=18-21&a=2") 
sleep(randint(2,4))

In [6]:
from selenium.webdriver.common.keys import Keys

In [6]:
sleep(3)
navegador.page_source # aqui esta el html
soup = BeautifulSoup(navegador.page_source)
soup

<html class="js canvas breakpoint-tablet-wide breakpoint-mobile no-touch csstransforms" lang="es"><head><link href="https://fonts.googleapis.com/css?family=Archivo:400,500|Arimo:400,500|Bitter:400,500|EB+Garamond:400,500|Lato|Libre+Baskervill|Libre+Franklin:400,500|Lora:400,500|Google+Sans:regular,medium:400,500|Material+Icons|Google+Symbols|Merriweather|Montserrat:400,500|Mukta:400,500|Muli:400,500|Nunito:400,500|Open+Sans:400,500,600|Open+Sans+Condensed:400,600|Oswald:500|Playfair+Display:400,500|Poppins:400,500|Raleway:400,500|Roboto:400,500|Roboto+Condensed:400,500|Roboto+Slab:400,500|Slabo+27px|Source+Sans+Pro|Ubuntu:400,500|Volkhov&amp;display=swap" rel="stylesheet"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="ie=edge" http-equiv="X-UA-Compatible"/>
<title>Aeropuerto de Barcelona El Prat Salidas hoy T1 y T2</title>
<meta content="Listado de salidas desde terminales T1 T2A T2B T2C del Aeropuerto de Barcelona El Prat (BCN-LEBL) hoy y mañana"

In [7]:
t = []
select = soup.find("select", {"name": "t"})

for opt in select.find_all("option"):
    valor = opt.get('value') 
    t.append(valor)
t = t[0:5] # noche scrapeo 
print(t)

['9-12', '12-15', '15-18', '18-21', '21-0']


In [8]:
import requests
from bs4 import BeautifulSoup
from time import sleep
from random import randint
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------------------------------------------------
# 1) CONFIGURAR SESIÓN ROBUSTA CON RETRY + USER AGENT
# ---------------------------------------------------------
session = requests.Session()

retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# ---------------------------------------------------------
# 2) LISTAS FINALES
# ---------------------------------------------------------
records_links = []   # links de vuelos individuales
destinos = []
aerolineas = []

puertas = []
vuelos = []
fechas = []
tipos = []
horas = []
estados = []

# ---------------------------------------------------------
# 3) SCRAPE DE LA LISTA PRINCIPAL
# ---------------------------------------------------------
def get_soup(url):
    """Descarga HTML con gestión de errores."""
    try:
        r = session.get(url, headers=headers, timeout=15)
        return BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(" Error descargando", url, str(e))
        return None


for f in t:
    print("Scrapeando franja:", f)

    url = f"https://www.aeropuertobarcelona-elprat.com/cast/salidas-aeropuerto-barcelona.html?t={f}&a=2"
    soup = get_soup(url)
    if not soup:
        continue

    flights = soup.find_all("div", class_="flightListRecord")

    for div in flights:

        time_block = div.find("div", class_="flightListTime")

        # Si no hay link, ignora TODO el registro completo
        if not (time_block and time_block.a):
            continue

        # ---- obtener link ----
        href = time_block.a.get("href")
        if href not in records_links:
            records_links.append(href)

        # ---- destino ----
        h = div.find("div", class_="flightListOtherAirport")
        if h:
            destino = "".join(h.find_all(string=True, recursive=False)).strip()
            destino = destino.lstrip("- ").strip()
            destinos.append(destino)
        else:
            destinos.append(None)

        # ---- aerolínea ----
        ae = div.find("a", class_="flightListFlightIDAirline")
        aerolineas.append(ae.get_text(strip=True) if ae else None)

# ---------------------------------------------------------
# 4) SCRAPE DE CADA VUELO INDIVIDUAL
# ---------------------------------------------------------
for lnk in records_links:

    full_url = f"https://www.aeropuertobarcelona-elprat.com{lnk}"
    print("Scrapeando vuelo:", full_url)

    soup = get_soup(full_url)
    if not soup:
        # rellenamos con None para no desalinear
        puertas.append(None)
        vuelos.append(None)
        fechas.append(None)
        tipos.append(None)
        horas.append(None)
        estados.append(None)
        continue

    p = soup.find("div", class_="panel__body txt--center")

    # Si no existe el bloque → todo None
    if not p:
        puertas.append(None)
        vuelos.append(None)
        fechas.append(None)
        tipos.append(None)
        horas.append(None)
        estados.append(None)
        continue

    # ---- puerta ----
    info = p.find("span", class_="numpuerta")
    puertas.append(info.get_text(strip=True) if info else None)

    # ---- número de vuelo ----
    v = p.find("div", class_="datodelvuelo nvuelo")
    vuelos.append(v.dd.get_text(strip=True) if v and v.dd else None)

    # ---- fecha ----
    fe = p.find("div", class_="datodelvuelo fecha")
    fechas.append(fe.dd.get_text(strip=True) if fe and fe.dd else None)

    # ---- tipo avión ----
    tip = p.find("div", class_="datodelvuelo avion")
    tipos.append(tip.dd.get_text(strip=True) if tip and tip.dd else None)

    # ---- hora ----
    ha = p.find("div", class_="datodelvuelo hora")
    horas.append(ha.dd.get_text(strip=True) if ha and ha.dd else None)

    # ---- estado ----
    es = p.find("div", class_="datodelvuelo status")
    estados.append(es.dd.get_text(strip=True) if es and es.dd else None)


# ---------------------------------------------------------
# 5) RESULTADOS
# ---------------------------------------------------------
print("Registros encontrados:", len(records_links))
print("Destinos:", len(destinos))
print("Aerolíneas:", len(aerolineas))
print("Vuelos individuales:", len(vuelos))


Scrapeando franja: 9-12
Scrapeando franja: 12-15
Scrapeando franja: 15-18
Scrapeando franja: 18-21
Scrapeando franja: 21-0
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-FR3166
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-W43938
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-W62048
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-FR2834
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-FR133
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-FR2239
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-FR3028
Scrapeando vuelo: https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-W61914
 Error descargando https://www.aeropuertobarcelona-elprat.com/cast/salida-vuelo-W61914 HTTPSConnectionPool(host='www.aeropuertobarcelona-elprat.com', port=443): Read timed out.
Scrapeando vu

In [9]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta
df_new = pd.DataFrame({'fecha_scrapeo':fechas, 'Name_company':aerolineas,'Num_flight':vuelos,
                   'Name_Destiny':destinos, 'Time_flight':horas, 'flight_state':estados,
                   'flight_capacity':'' ,'Aircraft':tipos, 'Gate': puertas
                    }) 
df_new.head(60)

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
0,13/Dic/2025,Ryanair,FR3166,Palermo (PMO),09:00,"En hora, Ha salido",,,U35
1,13/Dic/2025,Wizz Air Malta,W43938,Chisinau (KIV),09:00,"En hora, Ha salido",,,Y53
2,13/Dic/2025,Wizz Air,W62048,Krakow (KRK),09:05,"En hora, Ha salido",,,S20
3,13/Dic/2025,Ryanair,FR2834,Charleroi (CRL),09:15,"En hora, Ha salido",,,S23
4,13/Dic/2025,Ryanair,FR133,Berlin (BER),09:20,"En hora, Ha salido",,,U31
5,13/Dic/2025,Ryanair,FR2239,Marrakech (RAK),09:20,"En hora, Ha salido",,,W43
6,13/Dic/2025,Ryanair,FR3028,Ibiza (IBZ),09:20,"En hora, Ha salido",,,S21
7,None,Wizz Air,None,Vilnius (VNO),None,None,,None,None
8,13/Dic/2025,Ryanair,FR2889,,09:25,"En hora, Ha salido",,,U37
9,13/Dic/2025,Wizz Air Malta,W43176,Bucharest (OTP),09:25,"Retrasado, Ha salido 09:36",,,S22


In [46]:
#df_new = df_new[:-3]

In [10]:
df_new.shape

(117, 9)

In [11]:
df_new.replace("", np.nan, inplace=True)

C:\Users\linap\AppData\Local\Temp\ipykernel_5216\4020170196.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_new.replace("", np.nan, inplace=True)


In [26]:
df_new.tail(40)

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
134,None,Ryanair,None,NaN,None,None,NaN,None,None
135,08/Dic/2025,Air Arabia Maroc,3O378,Tanger (TNG),20:05,"En hora, Programado",NaN,NaN,NaN
136,08/Dic/2025,Ryanair,FR3807,Charleroi (CRL),20:05,"En hora, Programado",NaN,NaN,NaN
137,08/Dic/2025,Ryanair,FR6341,Rome (FCO),20:10,"En hora, Programado",NaN,NaN,NaN
138,08/Dic/2025,Ryanair,FR6597,Manchester (MAN),20:10,"En hora, Programado",NaN,NaN,NaN
139,08/Dic/2025,easyJet Europe,U21956,Milan (LIN),20:15,Programado,180.0,Airbus A320,None
140,08/Dic/2025,Wizz Air Malta,W42940,Vienna (VIE),20:20,"En hora, Programado",NaN,NaN,NaN
141,08/Dic/2025,easyJet Europe,EC1010,"Basel, Switzerland/Mulhouse (BSL)",20:30,"En hora, Programado",NaN,NaN,NaN
142,08/Dic/2025,T'Way Air,TW408,Seoul/Incheon (ICN),20:30,"En hora, Programado",NaN,NaN,NaN
143,08/Dic/2025,Volotea,V72135,Bordeaux (BOD),20:30,"En hora, Programado",NaN,NaN,NaN


In [12]:
df_new.isna().sum()

fecha_scrapeo        3
Name_company         1
Num_flight           3
Name_Destiny         7
Time_flight          3
flight_state         3
flight_capacity    117
Aircraft           107
Gate                47
dtype: int64

In [16]:
df_new[(df_new['fecha_scrapeo'].isna())] 

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
45,None,Wizz Air Malta,None,Milan (MXP),None,None,NaN,None,None
52,None,Ryanair,None,Frankfurt-Hahn (HHN),None,None,NaN,None,None


In [15]:
df_new.loc[
(df_new['fecha_scrapeo'].isna()) &
(df_new['Name_Destiny'] == 'Milan (MXP)'),
'Num_flight'] = 'W61914'
df_new.loc[df_new['Num_flight'] == 'W61914', 'Aircraft'] = "Airbus A321neo"
df_new.loc[df_new['Num_flight'] == 'W61914', 'flight_capacity'] = 239
df_new.loc[df_new['Num_flight'] == 'W61914', 'Time_flight'] = '09:20'
df_new.loc[df_new['Num_flight'] == 'W61914', 'fecha_scrapeo'] = '13/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'W61914', 'flight_state'] = 'Ha salido'
df_new.loc[df_new['Num_flight'] == 'W61914', 'Gate'] = 'U'

In [15]:
df_new.loc[
(df_new['fecha_scrapeo'].isna()) &
(df_new['Name_Destiny'] == 'Vilnius (VNO)'),
'Num_flight'] = 'W61914'
df_new.loc[df_new['Num_flight'] == 'W61914', 'Aircraft'] = "Airbus A321neo"
df_new.loc[df_new['Num_flight'] == 'W61914', 'flight_capacity'] = 239
df_new.loc[df_new['Num_flight'] == 'W61914', 'Time_flight'] = '09:20'
df_new.loc[df_new['Num_flight'] == 'W61914', 'fecha_scrapeo'] = '13/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'W61914', 'flight_state'] = 'Ha salido'
df_new.loc[df_new['Num_flight'] == 'W61914', 'Gate'] = 'U'

In [13]:
df_new[(df_new['Name_company'].isna())] 

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
60,13/Dic/2025,None,VF608,Ankara (ESB),14:50,"En hora, Ha salido",NaN,NaN,Y53


In [24]:
df_new.loc[
(df_new['fecha_scrapeo'].isna()) & (df_new['Name_Destiny'] == 'Bucharest (OTP)'),
'Num_flight'] = 'W43176'
df_new.loc[df_new['Num_flight'] == 'W43176', 'Aircraft'] = "Airbus A321neo"
df_new.loc[df_new['Num_flight'] == 'W43176', 'flight_capacity'] = 239
df_new.loc[df_new['Num_flight'] == 'W43176', 'Time_flight'] = '09:25'
df_new.loc[df_new['Num_flight'] == 'W43176', 'fecha_scrapeo'] = '08/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'W43176', 'Gate'] = 'S'
df_new.loc[df_new['Num_flight'] == 'W43176', 'flight_state'] = 'En hora'

df_new.loc[
(df_new['fecha_scrapeo'].isna()) & (df_new['Name_Destiny'] == 'Rome (FCO)'),
'Num_flight'] = 'W46018'
df_new.loc[df_new['Num_flight'] == 'W46018', 'Aircraft'] = "Airbus A321neo"
df_new.loc[df_new['Num_flight'] == 'W46018', 'flight_capacity'] = 239
df_new.loc[df_new['Num_flight'] == 'W46018', 'Time_flight'] = '10:05'
df_new.loc[df_new['Num_flight'] == 'W46018', 'fecha_scrapeo'] = '08/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'W46018', 'Gate'] = 'U'
df_new.loc[df_new['Num_flight'] == 'W46018', 'flight_state'] = 'En hora'

df_new.loc[
(df_new['fecha_scrapeo'].isna()) & (df_new['Name_Destiny'] == 'Manchester (MAN)'),
'Num_flight'] = 'U22002'
df_new.loc[df_new['Num_flight'] == 'U22002', 'Aircraft'] = "Airbus A320"
df_new.loc[df_new['Num_flight'] == 'U22002', 'flight_capacity'] = 180
df_new.loc[df_new['Num_flight'] == 'U22002', 'Time_flight'] = '11:20'
df_new.loc[df_new['Num_flight'] == 'U22002', 'fecha_scrapeo'] = '08/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'U22002', 'Gate'] = 'M'
df_new.loc[df_new['Num_flight'] == 'U22002', 'flight_state'] = 'Retrasado'

df_new.loc[
(df_new['fecha_scrapeo'].isna()) & (df_new['Name_Destiny'] == 'Cologne-Bonn (CGN)'),
'Num_flight'] = 'EW521'
df_new.loc[df_new['Num_flight'] == 'EW521', 'Aircraft'] = "Airbus A320"
df_new.loc[df_new['Num_flight'] == 'EW521', 'flight_capacity'] = 180
df_new.loc[df_new['Num_flight'] == 'EW521', 'Time_flight'] = '13:25'
df_new.loc[df_new['Num_flight'] == 'EW521', 'fecha_scrapeo'] = '08/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'EW521', 'Gate'] = 'U'
df_new.loc[df_new['Num_flight'] == 'EW521', 'flight_state'] = 'Retrasado'

df_new.loc[
(df_new['fecha_scrapeo'].isna()) & (df_new['Name_Destiny'] == 'Milan (LIN)'),
'Num_flight'] = 'U21956'
df_new.loc[df_new['Num_flight'] == 'U21956', 'Aircraft'] = "Airbus A320"
df_new.loc[df_new['Num_flight'] == 'U21956', 'flight_capacity'] = 180
df_new.loc[df_new['Num_flight'] == 'U21956', 'Time_flight'] = '20:15'
df_new.loc[df_new['Num_flight'] == 'U21956', 'fecha_scrapeo'] = '08/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'EC1956', 'flight_state'] = 'Programado'

In [17]:
df_new.loc[df_new['Num_flight'] == 'LW1342', 'Aircraft'] = "Airbus A320"
df_new.loc[df_new['Num_flight'] == 'LW1342', 'Name_Destiny'] = "Palma de Mallorca (PMI)"
df_new.loc[df_new['Num_flight'] == 'LW1342', 'flight_capacity'] = 180

df_new.loc[df_new['Num_flight'] == 'FR1165', 'Aircraft'] = "Boeing 737-800"
df_new.loc[df_new['Num_flight'] == 'FR1165', 'Name_Destiny'] = "Sevilla (SVQ)"
df_new.loc[df_new['Num_flight'] == 'FR1165', 'flight_capacity'] = 189

df_new.loc[df_new['Num_flight'] == 'FR2495', 'Aircraft'] = "Boeing 737-800"
df_new.loc[df_new['Num_flight'] == 'FR2495', 'Name_Destiny'] = "Palma de Mallorca (PMI)"
df_new.loc[df_new['Num_flight'] == 'FR2495', 'flight_capacity'] = 189

df_new.loc[df_new['Num_flight'] == 'FR483', 'Aircraft'] = "Boeing 737 MAX 8"
df_new.loc[df_new['Num_flight'] == 'FR483', 'Name_Destiny'] = "Malaga (AGP)"
df_new.loc[df_new['Num_flight'] == 'FR483', 'flight_capacity'] = 197

df_new.loc[df_new['Num_flight'] == 'FR2889', 'Aircraft'] = "Boeing 737-800"
df_new.loc[df_new['Num_flight'] == 'FR2889', 'Name_Destiny'] = "Palma de Mallorca (PMI)"
df_new.loc[df_new['Num_flight'] == 'FR2889', 'flight_capacity'] = 189

df_new.loc[df_new['Num_flight'] == 'V73735', 'Aircraft'] = "Airbus A320"
df_new.loc[df_new['Num_flight'] == 'V73735', 'Name_Destiny'] = "Murcia (RMU)"
df_new.loc[df_new['Num_flight'] == 'V73735', 'flight_capacity'] = 180

df_new.loc[df_new['Num_flight'] == 'LW2896', 'Aircraft'] = "Airbus A320"
df_new.loc[df_new['Num_flight'] == 'LW2896', 'Name_Destiny'] = "Palma de Mallorca (PMI)"
df_new.loc[df_new['Num_flight'] == 'LW2896', 'flight_capacity'] = 180

df_new.loc[df_new['Num_flight'] == 'FR2402', 'Aircraft'] = "Boeing 737-800"
df_new.loc[df_new['Num_flight'] == 'FR2402', 'Name_Destiny'] = "Sevilla (SVQ)"
df_new.loc[df_new['Num_flight'] == 'FR2402', 'flight_capacity'] = 189

In [16]:
df_new[(df_new['fecha_scrapeo'].isna())] 

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
24,None,Wizz Air Malta,None,Bucharest (OTP),None,None,NaN,None,None
38,None,Wizz Air Malta,None,Rome (FCO),None,None,NaN,None,None
52,None,Easyjet,None,Manchester (MAN),None,None,NaN,None,None
72,None,Eurowings,None,Cologne-Bonn (CGN),None,None,NaN,None,None
134,None,Ryanair,None,NaN,None,None,NaN,None,None
139,None,easyJet Europe,None,Milan (LIN),None,None,NaN,None,None


In [28]:
df_new.loc[
(df_new['fecha_scrapeo'].isna()),
'Num_flight'] = 'FR3070'
df_new.loc[df_new['Num_flight'] == 'FR3070', 'Aircraft'] = "Boeing 737-800"
df_new.loc[df_new['Num_flight'] == 'FR3070', 'flight_capacity'] = 189
df_new.loc[df_new['Num_flight'] == 'FR3070', 'Time_flight'] = '14:40'
df_new.loc[df_new['Num_flight'] == 'FR3070', 'Name_Destiny'] = 'Palma De Mallorca (PMI)'
df_new.loc[df_new['Num_flight'] == 'FR3070', 'fecha_scrapeo'] = '06/Dic/2025'
df_new.loc[df_new['Num_flight'] == 'FR3070', 'Gate'] = 'S'
df_new.loc[df_new['Num_flight'] == 'FR3070', 'flight_state'] = 'En hora'

In [60]:
df_new.loc[
(df_new['fecha_scrapeo'].isna()) & 
(df_new['Name_Destiny'] == 'Milan (MXP)'), 
'Num_flight'] = 'W46330'
df_new.loc[df_new['Num_flight'] == 'W46330', 'Aircraft'] = "Airbus A321neo"
df_new.loc[df_new['Num_flight'] == 'W46330', 'flight_capacity'] = 239
df_new.loc[df_new['Num_flight'] == 'W46330', 'Time_flight'] = '13:10'
df_new.loc[df_new['Num_flight'] == 'W46330', 'flight_state'] = 'Retrasado'
df_new.loc[df_new['Num_flight'] == 'W46330', 'Gate'] = 'U'
df_new.loc[df_new['Num_flight'] == 'W46330', 'fecha_scrapeo'] = '05/Dic/2025'

In [32]:
df_new[(df_new['Name_Destiny'].isna())]

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate


In [171]:
#df_new = df_new.drop(21)
#df_new = df_new.drop(119)
#df_new.reset_index(drop=True, inplace=True)

In [168]:
df_new[(df_new['fecha_scrapeo'].isna())]

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
67,None,Ryanair,None,NaN,None,None,NaN,None,None


In [165]:
df_new.loc[df_new['Name_Destiny'] == 'Oslo (OSL)', 'Num_flight'] = "DY1741"

In [180]:
df_new.loc[df_new['Num_flight'] == 'DY1741', 'Aircraft'] = "Boeing 737 MAX 8"
df_new.loc[df_new['Num_flight'] == 'DY1741', 'flight_capacity'] = 189
df_new.loc[df_new['Num_flight'] == 'DY1741', 'Gate'] = 'S'
df_new.loc[df_new['Num_flight'] == 'DY1741', 'Time_flight'] = '11:15'
df_new.loc[df_new['Num_flight'] == 'DY1741', 'flight_state'] = 'En hora'
df_new.loc[df_new['Num_flight'] == 'DY1741', 'fecha_scrapeo'] = '30/Nov/2025'

In [189]:
df_new.loc[60:70]

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate
60,30/Nov/2025,Volotea,V71445,Olbia (OLB),13:05,"En hora, Ha salido",NaN,NaN,S20
61,30/Nov/2025,Wizz Air Malta,W46330,Milan (MXP),13:10,"Retrasado, Ha salido 13:27",NaN,NaN,S24
62,30/Nov/2025,Ryanair,FR3121,Beauvais (BVA),13:20,"En hora, Ha salido",NaN,NaN,U31
63,30/Nov/2025,Easyjet,U28056,London (LGW),13:25,"En hora, Ha salido",NaN,NaN,M6
64,30/Nov/2025,easyJet Europe,EC1380,Geneva (GVA),13:40,"En hora, Ha salido",NaN,NaN,R9
65,30/Nov/2025,Ryanair,FR9811,London (STN),13:40,"En hora, Ha salido",NaN,NaN,W45
66,30/Nov/2025,Ryanair,FR483,Malaga (AGP),13:50,En hora,189.0,Boeing 737-800,S
67,30/Nov/2025,Air Baltic,BT684,Riga (RIX),14:05,"En hora, Programado",NaN,NaN,S23
68,30/Nov/2025,easyJet Europe,EC7153,Milan (MXP),14:05,"En hora, Ha salido",NaN,NaN,R10
69,30/Nov/2025,Ryanair,FR1342,Palma de Mallorca (PMI),14:05,"En hora, Ha salido",189.0,Boeing 737-800,U35


In [105]:
df_new.loc[
(df_new['Name_company'] == 'Volotea') & 
(df_new['Num_flight'].isna()),
'Num_flight'
] = 'V73735' 

In [109]:
df_new.loc[df_new['Num_flight'] == 'V73735', 'Time_flight'] = "15:15"
df_new.loc[df_new['Num_flight'] == 'V73735', 'Gate'] = "S"
df_new.loc[df_new['Num_flight'] == 'V73735', 'flight_state'] = "Ha salido"
df_new.loc[df_new['Num_flight'] == 'V73735', 'Aircraft'] = "Airbus A319"
df_new.loc[df_new['Num_flight'] == 'V73735', 'Name_Destiny'] = "Corvera (RMU)"

In [101]:
duplicados = df_new[df_new.duplicated(subset=['Num_flight', 'Time_flight'], keep=False)]
duplicados.sort_values(['Num_flight', 'Time_flight'])

,fecha_scrapeo,Name_company,Num_flight,Name_Destiny,Time_flight,flight_state,flight_capacity,Aircraft,Gate


In [44]:
# W61476  SI SE ES LA COMPANIA TAL Y EL DESTINO CUAL 
#PON EL NUM DE VUELO, QUE NO SE HA ESCRAPEADO BIEN DEBIDO A BLOQUEO
df_new.loc[
(df_new['Name_company'] == 'Wizz Air') & 
(df_new['Name_Destiny'] == 'Warsaw (WAW)') &
(df_new['Num_flight'].isna()),
'Num_flight'
] = 'W61476'  

In [68]:
df_new.loc[df_new['Num_flight'] == 'LW2889', 'Name_Destiny'] = "Palma de Mallorca (PMI)"
df_new.loc[df_new['Num_flight'] == 'LW2889', 'Aircraft'] = "Airbus A320"

In [60]:
df.duplicated().sum()

0

In [121]:
df_new.isna().sum()

fecha_scrapeo        0
Name_company         0
Num_flight           0
Name_Destiny         0
Time_flight          0
flight_state         0
flight_capacity    168
Aircraft           142
Gate                83
dtype: int64

In [119]:
df_new.reset_index(drop=True, inplace=True)

In [120]:
df_new.shape

(168, 9)

In [31]:
#n = len(df_f['flight_state'])  # número de estados nuevos
#df.iloc[-n:, df.columns.get_loc('flight_state')] = df_f['flight_state'].values

In [39]:
#df.to_csv("vuelos_bcn.csv", index=False)

In [33]:
# Asegúrate de que el índice está correcto
#df_n = df_new.reset_index(drop=True)  # opcional: genera índice 0,1,2,...

# Cargar CSV limpio
df_csv = pd.read_csv("vuelos_bcn_sindup.csv")

# Añadir el dataset completo
df_fin = pd.concat([df_csv, df_new], ignore_index=True)

# Guardar CSV final
df_fin.to_csv("vuelos_bcn_sindup.csv", index=False)
